## Consultas - Saldos OC

#### 1.Carregar tabela da camada silver

In [ ]:
import sys
sys.path.append("/app")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable


spark = create_spark_session("saldos_oc")
config = load_config()

# ler tabelas silver
df_produtos_estoques_movimentacoes = spark.read.format("delta").load(
    f"data/silver/produtos_estoques_movimentacoes"
)
df_rel_compras_ordens_referenciadas_saldos = spark.read.format("delta").load(
    f"data/silver/rel_compras_ordens_referenciadas_saldos"
)
df_compras_ordens_itens = spark.read.format("delta").load(
    f"data/silver/compras_ordens_itens"
)
df_pedidos_itens = spark.read.format("delta").load(
    f"data/silver/pedidos_itens"
)
df_pedidos = spark.read.format("delta").load(
    f"data/silver/pedidos"
)
df_produtos = spark.read.format("delta").load(
    f"data/silver/produtos"
)
df_estoques = spark.read.format("delta").load(
    f"data/silver/estoques"
)
df_pessoas = spark.read.format("delta").load(
    f"data/silver/pessoas"
)

# Registrar como temporary views
df_produtos_estoques_movimentacoes.createOrReplaceTempView("produtos_estoques_movimentacoes")
df_rel_compras_ordens_referenciadas_saldos.createOrReplaceTempView("rel_compras_ordens_referenciadas_saldos")
df_compras_ordens_itens.createOrReplaceTempView("compras_ordens_itens")
df_pedidos_itens.createOrReplaceTempView("pedidos_itens")
df_pedidos.createOrReplaceTempView("pedidos")
df_produtos.createOrReplaceTempView("produtos")
df_estoques.createOrReplaceTempView("estoques")
df_pessoas.createOrReplaceTempView("pessoas")


#### 2.Executar transformação

In [ ]:
df_gold = spark.sql("""
WITH ult_mov AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY produto_id, empresa_id
            ORDER BY data_lancamento DESC, hora_lancamento DESC, id DESC
        ) as rn
    FROM produtos_estoques_movimentacoes
),

mov_sp AS (
    SELECT produto_id, saldo AS Deposito_SP
    FROM ult_mov
    WHERE empresa_id = 1 AND rn = 1
),

mov_sc AS (
    SELECT produto_id, saldo AS Deposito_SC
    FROM ult_mov
    WHERE empresa_id = 210 AND rn = 1
),

saldo_oc AS (
    SELECT 
        sld.produto_id,
        sld.empresa_id,
        SUM(sld.saldo) as saldo_ordem_compra
    FROM rel_compras_ordens_referenciadas_saldos sld
    LEFT JOIN compras_ordens_itens coi 
        ON coi.ordem_compra_id = sld.ordem_compra_id 
        AND coi.produto_id = sld.produto_id 
        AND sld.id_item = coi.id
    WHERE sld.saldo <> 0
    GROUP BY sld.produto_id, sld.empresa_id
),

base AS (
    SELECT 
        p.id AS OV,
        date_format(p.data_emissao, 'dd/MM/yyyy') AS data_emissao,
        i.codigo_pedido,
        p.cliente_id,
        cliente.razao_social AS cliente_nome,
        p.empresa_id,
        i.numero_item,
        i.produto_id,
        m.fornecedor_id,
        fornecedor.razao_social AS fornecedor_nome,
        m.codigo_identificacao_interno,
        m.nome,
        i.quantidade,
        COALESCE(sp.Deposito_SP, 0) + COALESCE(sc.Deposito_SC, 0) AS estoque,
        COALESCE(sp.Deposito_SP, 0) AS Deposito_SP,
        COALESCE(sc.Deposito_SC, 0) AS Deposito_SC,
        e.minimo AS estoque_minimo,
        e.maximo AS estoque_maximo,
        date_format(i.data_entrega, 'dd/MM/yyyy') AS data_entrega,
        COALESCE(soc.saldo_ordem_compra, 0) AS saldo_ordem_compra
    FROM pedidos_itens i
    LEFT JOIN pedidos p ON p.id = i.pedido_id
    LEFT JOIN produtos m ON m.id = i.produto_id
    LEFT JOIN estoques e 
        ON e.material_id = i.produto_id 
        AND e.empresa_id = p.empresa_id
    LEFT JOIN pessoas cliente ON cliente.id = p.cliente_id
    LEFT JOIN pessoas fornecedor ON fornecedor.id = m.fornecedor_id
    LEFT JOIN mov_sp sp ON sp.produto_id = i.produto_id
    LEFT JOIN mov_sc sc ON sc.produto_id = i.produto_id
    LEFT JOIN saldo_oc soc 
        ON soc.produto_id = i.produto_id 
        AND soc.empresa_id = p.empresa_id
    WHERE p.status = 1
)

SELECT *,
    (saldo_ordem_compra + estoque)
    - SUM(quantidade) OVER (
        PARTITION BY produto_id 
        ORDER BY data_entrega 
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS saldo
FROM base
""")

#### 3.Armazenar dados na camada gold

In [ ]:
path = "data/gold/pedidos_ordem_compras"

df_gold.write.format("delta").mode("overwrite").save(path)